# Pipeline de Alfabetização no Brasil — Notebook 1: Setup e Ingestão Bronze

**Tech Challenge Fase 2 — FIAP POSTECH**  
**Disciplina:** Arquitetura de Big Data  

---

## Objetivo

Este notebook realiza:
1. Configuração do ambiente AWS (S3 + Glue + Athena)
2. Ingestão dos dados brutos (camada **Bronze**) a partir dos arquivos INEP locais
3. Upload para S3 no formato Parquet, preservando o histórico completo

## Arquitetura Medalhão

```
Fontes INEP (CSV)  ──►  Bronze (S3/Parquet)  ──►  Silver  ──►  Gold  ──►  Athena
```

## Fontes de Dados

| Arquivo | Descrição | Linhas |
|---|---|---|
| `br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv` | Meta nacional de alfabetização | ~3 |
| `br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv` | Metas por UF | ~54 |
| `br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv` | Metas por município | ~10.704 |
| `br_inep_avaliacao_alfabetizacao_uf.csv` | Indicadores por UF | ~145 |
| `br_inep_avaliacao_alfabetizacao_municipio.csv` | Indicadores por município | ~23.995 |

## 1. Instalação e Imports

In [2]:
# Instala dependências se necessário
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "boto3", "pandas", "pyarrow", "python-dotenv"], check=True)
print("Dependências OK")

0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


Dependências OK


In [3]:
import os
import io
import logging
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import boto3
from botocore.exceptions import ClientError

# Carrega .env se existir
try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"))
except ImportError:
    pass

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)
print("Imports OK")

Imports OK


## 2. Configuração

In [4]:
# ── Configurações (ajuste conforme seu ambiente) ──────────────────────────────
S3_BUCKET    = os.getenv("S3_BUCKET_NAME",       "tech-challenge-alfabetizacao-01")
AWS_REGION   = os.getenv("AWS_DEFAULT_REGION",   "us-east-1")
GLUE_DB      = os.getenv("GLUE_DATABASE",        "alfabetizacao_db")
ATHENA_OUT   = os.getenv("ATHENA_OUTPUT_LOCATION", f"s3://{S3_BUCKET}/athena-results/")

# Caminho local dos dados INEP
INEP_DIR = Path(os.getenv(
    "INEP_DATA_DIR",
    "../../../Desktop/FIAP - Ciências de Dados com IA/Tech Challenges/Fase 2/Dados INEP"
)).expanduser()

# Prefixos S3
BRONZE_PREFIX = "bronze"
RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")

print(f"Bucket  : {S3_BUCKET}")
print(f"Região  : {AWS_REGION}")
print(f"Glue DB : {GLUE_DB}")
print(f"Run TS  : {RUN_TS}")
print(f"INEP dir: {INEP_DIR}")

Bucket  : tech-challenge-alfabetizacao-01
Região  : us-east-1
Glue DB : alfabetizacao_db
Run TS  : 20260710T205911
INEP dir: ../../../Desktop/FIAP - Ciências de Dados com IA/Tech Challenges/Fase 2/Dados INEP


## 3. Setup da Infraestrutura AWS

In [5]:
s3_client   = boto3.client("s3", region_name=AWS_REGION)
glue_client = boto3.client("glue", region_name=AWS_REGION)
athena_client = boto3.client("athena", region_name=AWS_REGION)


def create_bucket(bucket: str, region: str):
    """Cria bucket S3 idempotente."""
    try:
        if region == "us-east-1":
            s3_client.create_bucket(Bucket=bucket)
        else:
            s3_client.create_bucket(
                Bucket=bucket,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        # Bloqueia acesso público (segurança)
        s3_client.put_public_access_block(
            Bucket=bucket,
            PublicAccessBlockConfiguration={
                "BlockPublicAcls": True, "IgnorePublicAcls": True,
                "BlockPublicPolicy": True, "RestrictPublicBuckets": True
            }
        )
        logger.info("Bucket criado: s3://%s", bucket)
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
            logger.info("Bucket já existe: s3://%s", bucket)
        else:
            raise


def set_lifecycle_policy(bucket: str):
    """FinOps: move dados Bronze para S3-IA após 90 dias."""
    policy = {
        "Rules": [{
            "ID": "bronze-to-ia",
            "Filter": {"Prefix": "bronze/"},
            "Status": "Enabled",
            "Transitions": [{"Days": 90, "StorageClass": "STANDARD_IA"}]
        }]
    }
    s3_client.put_bucket_lifecycle_configuration(
        Bucket=bucket, LifecycleConfiguration=policy
    )
    logger.info("Lifecycle policy: bronze/ → STANDARD_IA após 90 dias")


def create_glue_database(db: str):
    try:
        glue_client.create_database(
            DatabaseInput={"Name": db, "Description": "Alfabetização Brasil — Tech Challenge Fase 2"}
        )
        logger.info("Glue database criado: %s", db)
    except ClientError as e:
        if e.response["Error"]["Code"] == "AlreadyExistsException":
            logger.info("Glue database já existe: %s", db)
        else:
            raise


def create_athena_workgroup(wg: str, output: str):
    try:
        athena_client.create_work_group(
            Name=wg,
            Configuration={
                "ResultConfiguration": {"OutputLocation": output},
                "EnforceWorkGroupConfiguration": True,
                "PublishCloudWatchMetricsEnabled": True,
                # FinOps: limita scan a 1 GB por query
                "BytesScannedCutoffPerQuery": 1_073_741_824
            },
            Description="Workgroup alfabetização"
        )
        logger.info("Athena workgroup criado: %s", wg)
    except ClientError as e:
        if "already exists" in str(e).lower() or "InvalidRequestException" in str(e):
            logger.info("Athena workgroup já existe: %s", wg)
        else:
            raise


# Executa setup (descomente quando tiver credenciais AWS)
# create_bucket(S3_BUCKET, AWS_REGION)
# set_lifecycle_policy(S3_BUCKET)
# create_glue_database(GLUE_DB)
# create_athena_workgroup("alfabetizacao", ATHENA_OUT)

print("Funções de setup definidas.")
print("Descomente as linhas acima para executar o setup real na AWS.")

2026-07-10 17:59:54,009 [INFO] Found credentials in environment variables.


Funções de setup definidas.
Descomente as linhas acima para executar o setup real na AWS.


## 4. Carregamento dos Dados INEP (Fontes)

In [6]:
# Mapeamento: nome lógico → arquivo CSV INEP
INEP_FILES = {
    "meta_brasil"    : "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_brasil.csv",
    "meta_uf"        : "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_uf.csv",
    "meta_municipio" : "br_inep_avaliacao_alfabetizacao_meta_alfabetizacao_municipio.csv",
    "indicador_uf"   : "br_inep_avaliacao_alfabetizacao_uf.csv",
    "indicador_municipio": "br_inep_avaliacao_alfabetizacao_municipio.csv",
}


def load_csv(name: str, filename: str) -> pd.DataFrame:
    """Carrega CSV do INEP e adiciona metadados de ingestão."""
    path = INEP_DIR / filename
    if not path.exists():
        logger.warning("Arquivo não encontrado: %s", path)
        return pd.DataFrame()
    df = pd.read_csv(path, encoding="utf-8", dtype=str)
    # Metadados Bronze (padrão das aulas de Arquitetura de Big Data)
    df["_fonte"]         = name
    df["_arquivo_origem"] = filename
    df["_data_ingestao"] = RUN_TS
    logger.info("%-25s: %d linhas, %d colunas", name, len(df), len(df.columns))
    return df


raw_datasets = {}
for name, filename in INEP_FILES.items():
    raw_datasets[name] = load_csv(name, filename)

print("\nDatasets carregados:")
for name, df in raw_datasets.items():
    print(f"  {name:30s}: {len(df):>6,} linhas")

2026-07-10 18:00:48,377 [INFO] meta_brasil              : 3 linhas, 14 colunas
2026-07-10 18:00:48,386 [INFO] meta_uf                  : 54 linhas, 15 colunas
2026-07-10 18:00:48,402 [INFO] meta_municipio           : 10704 linhas, 16 colunas
2026-07-10 18:00:48,410 [INFO] indicador_uf             : 145 linhas, 18 colunas
2026-07-10 18:00:48,442 [INFO] indicador_municipio      : 23995 linhas, 18 colunas



Datasets carregados:
  meta_brasil                   :      3 linhas
  meta_uf                       :     54 linhas
  meta_municipio                : 10,704 linhas
  indicador_uf                  :    145 linhas
  indicador_municipio           : 23,995 linhas


## 5. Exploração dos Dados Brutos (EDA Bronze)

In [7]:
print("=" * 60)
print("META BRASIL")
print("=" * 60)
display(raw_datasets["meta_brasil"].drop(columns=["_fonte","_arquivo_origem","_data_ingestao"]))

META BRASIL


,ano,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2025,Pública,66,60,64,67,71,74,77,80,88
1,2024,Pública,59.2,59.9,63.77,67.47,70.97,74.23,77.24,80,87.37
2,2023,Pública,55.9,59.9,63.77,67.47,70.97,74.23,77.24,80,86


In [8]:
print("=" * 60)
print("META UF — primeiras linhas")
print("=" * 60)
display(raw_datasets["meta_uf"]
        .drop(columns=["_fonte","_arquivo_origem","_data_ingestao"])
        .head(10))

META UF — primeiras linhas


,ano,sigla_uf,rede,taxa_alfabetizacao,meta_alfabetizacao_2024,meta_alfabetizacao_2025,meta_alfabetizacao_2026,meta_alfabetizacao_2027,meta_alfabetizacao_2028,meta_alfabetizacao_2029,meta_alfabetizacao_2030,percentual_participacao
0,2024,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,RR,Pública,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,AC,Pública,NaN,NaN,56.9,62.2,67.3,72,76.2,80,NaN
3,2024,AC,Pública,51.38,NaN,56.9,62.2,67.3,72,76.2,80,80.87
4,2023,DF,Pública,NaN,NaN,63.1,67,70.6,74,77.1,80,NaN
5,2024,DF,Pública,59.13,NaN,63.1,67,70.6,74,77.1,80,79.83
6,2023,SE,Pública,31.3,38.3,45.9,53.6,61.2,68.3,74.6,80,88.34
7,2024,SE,Pública,38.39,38.3,45.9,53.6,61.2,68.3,74.6,80,92.84
8,2024,BA,Pública,35.96,43.4,50.2,57.1,63.6,69.8,75.2,80,90.04
9,2023,BA,Pública,36.8,43.4,50.2,57.1,63.6,69.8,75.2,80,84.53


In [9]:
print("=" * 60)
print("INDICADOR MUNICÍPIO — schema e estatísticas")
print("=" * 60)
df_ind = raw_datasets["indicador_municipio"].drop(columns=["_fonte","_arquivo_origem","_data_ingestao"])
print("Colunas:", df_ind.columns.tolist())
print()
display(df_ind.head(5))
print()
print("Anos disponíveis:", sorted(df_ind["ano"].unique()))
print("Municípios únicos:", df_ind["id_municipio"].nunique())
print("Nulos por coluna:")
print(df_ind.isnull().sum())

INDICADOR MUNICÍPIO — schema e estatísticas
Colunas: ['ano', 'id_municipio', 'serie', 'rede', 'taxa_alfabetizacao', 'media_portugues', 'proporcao_aluno_nivel_0', 'proporcao_aluno_nivel_1', 'proporcao_aluno_nivel_2', 'proporcao_aluno_nivel_3', 'proporcao_aluno_nivel_4', 'proporcao_aluno_nivel_5', 'proporcao_aluno_nivel_6', 'proporcao_aluno_nivel_7', 'proporcao_aluno_nivel_8']



,ano,id_municipio,serie,rede,taxa_alfabetizacao,media_portugues,proporcao_aluno_nivel_0,proporcao_aluno_nivel_1,proporcao_aluno_nivel_2,proporcao_aluno_nivel_3,proporcao_aluno_nivel_4,proporcao_aluno_nivel_5,proporcao_aluno_nivel_6,proporcao_aluno_nivel_7,proporcao_aluno_nivel_8
0,2023,1100031,2,3,69.1,767.8763,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2023,1100072,2,3,58.2,747.8918,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2023,1100189,2,5,69.73,762.4062,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2023,1101609,2,3,50.7,745.6802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2023,1101807,2,3,55.69,752.3724,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Anos disponíveis: ['2023', '2024']
Municípios únicos: 5550
Nulos por coluna:
ano                            0
id_municipio                   0
serie                          0
rede                           0
taxa_alfabetizacao             0
media_portugues                0
proporcao_aluno_nivel_0    11547
proporcao_aluno_nivel_1    11547
proporcao_aluno_nivel_2    11547
proporcao_aluno_nivel_3    11547
proporcao_aluno_nivel_4    11547
proporcao_aluno_nivel_5    11547
proporcao_aluno_nivel_6    11547
proporcao_aluno_nivel_7    11547
proporcao_aluno_nivel_8    11547
dtype: int64


## 6. Upload Bronze para S3 (Parquet)

In [10]:
def upload_to_s3_bronze(df: pd.DataFrame, source_name: str) -> str:
    """Serializa DataFrame como Parquet e faz upload para S3 Bronze."""
    if df.empty:
        logger.warning("DataFrame vazio para '%s' — pulando upload.", source_name)
        return ""
    
    buffer = io.BytesIO()
    df.to_parquet(buffer, index=False, engine="pyarrow")
    buffer.seek(0)
    
    # Particionamento por data de ingestão (FinOps: facilita expurgo)
    s3_key = f"{BRONZE_PREFIX}/{source_name}/run_ts={RUN_TS}/{source_name}.parquet"
    
    s3_client.put_object(
        Bucket=S3_BUCKET,
        Key=s3_key,
        Body=buffer.getvalue(),
        ContentType="application/octet-stream"
    )
    logger.info("Upload OK: s3://%s/%s (%d linhas)", S3_BUCKET, s3_key, len(df))
    return s3_key


print("Função de upload Bronze definida.")
print()
print("Para executar o upload real, rode a célula abaixo com credenciais AWS configuradas.")

Função de upload Bronze definida.

Para executar o upload real, rode a célula abaixo com credenciais AWS configuradas.


In [11]:
# ── Execute esta célula com credenciais AWS ──────────────────────────────────
# (se não tiver AWS, os dados ficam em memória e passam para o próximo notebook)

uploaded_keys = {}
USE_AWS = os.getenv("USE_AWS", "false").lower() == "true"

if USE_AWS:
    for name, df in raw_datasets.items():
        key = upload_to_s3_bronze(df, name)
        if key:
            uploaded_keys[name] = key
    print(f"\nUpload concluído: {len(uploaded_keys)} tabelas enviadas para s3://{S3_BUCKET}/{BRONZE_PREFIX}/")
else:
    print("[MODO LOCAL] USE_AWS=false — dados mantidos em memória para continuar o pipeline.")
    print("Para usar S3 real: defina USE_AWS=true no .env")

# Em modo local, salva Parquet localmente (serve de base para Silver)
LOCAL_BRONZE_DIR = Path("../data/bronze")
LOCAL_BRONZE_DIR.mkdir(parents=True, exist_ok=True)

for name, df in raw_datasets.items():
    out = LOCAL_BRONZE_DIR / f"{name}.parquet"
    df.to_parquet(out, index=False, engine="pyarrow")
    print(f"  Salvo localmente: {out} ({len(df)} linhas)")

2026-07-10 18:01:28,567 [INFO] Upload OK: s3://tech-challenge-alfabetizacao-01/bronze/meta_brasil/run_ts=20260710T205911/meta_brasil.parquet (3 linhas)
2026-07-10 18:01:28,895 [INFO] Upload OK: s3://tech-challenge-alfabetizacao-01/bronze/meta_uf/run_ts=20260710T205911/meta_uf.parquet (54 linhas)
2026-07-10 18:01:29,570 [INFO] Upload OK: s3://tech-challenge-alfabetizacao-01/bronze/meta_municipio/run_ts=20260710T205911/meta_municipio.parquet (10704 linhas)
2026-07-10 18:01:29,901 [INFO] Upload OK: s3://tech-challenge-alfabetizacao-01/bronze/indicador_uf/run_ts=20260710T205911/indicador_uf.parquet (145 linhas)
2026-07-10 18:01:30,603 [INFO] Upload OK: s3://tech-challenge-alfabetizacao-01/bronze/indicador_municipio/run_ts=20260710T205911/indicador_municipio.parquet (23995 linhas)



Upload concluído: 5 tabelas enviadas para s3://tech-challenge-alfabetizacao-01/bronze/
  Salvo localmente: ../data/bronze/meta_brasil.parquet (3 linhas)
  Salvo localmente: ../data/bronze/meta_uf.parquet (54 linhas)
  Salvo localmente: ../data/bronze/meta_municipio.parquet (10704 linhas)
  Salvo localmente: ../data/bronze/indicador_uf.parquet (145 linhas)
  Salvo localmente: ../data/bronze/indicador_municipio.parquet (23995 linhas)


## 7. Verificação da Camada Bronze

In [12]:
print("Resumo da Ingestão Bronze")
print("-" * 50)
total_rows = 0
for name, df in raw_datasets.items():
    n_rows    = len(df)
    n_cols    = len(df.columns) - 3  # desconta metadados
    n_nulls   = df.isnull().sum().sum()
    total_rows += n_rows
    print(f"  {name:30s}: {n_rows:>7,} linhas | {n_cols:>2} colunas | {n_nulls:>6,} nulos")

print("-" * 50)
print(f"  {'TOTAL':30s}: {total_rows:>7,} linhas")
print()
print("Próximo passo: execute o notebook 02_silver_transformation.ipynb")

Resumo da Ingestão Bronze
--------------------------------------------------
  meta_brasil                   :       3 linhas | 11 colunas |      0 nulos
  meta_uf                       :      54 linhas | 12 colunas |     26 nulos
  meta_municipio                :  10,704 linhas | 13 colunas |    600 nulos
  indicador_uf                  :     145 linhas | 15 colunas |    630 nulos
  indicador_municipio           :  23,995 linhas | 15 colunas | 103,923 nulos
--------------------------------------------------
  TOTAL                         :  34,901 linhas

Próximo passo: execute o notebook 02_silver_transformation.ipynb


---
## Decisões Arquiteturais — Bronze Layer

| Decisão | Escolha | Motivo |
|---|---|---|
| Formato de armazenamento | **Parquet** | 60-80% menor que CSV; columnar otimiza leitura parcial no Athena |
| Sem transformações | Dados brutos preservados | Bronze = histórico fidedigno; mudanças de schema não destroem o passado |
| Metadados `_data_ingestao` / `_fonte` | Todos os registros | Rastreabilidade para auditoria e reprocessamento |
| Lifecycle S3-IA após 90 dias | Apenas Bronze | Silver/Gold são consultados com frequência; Bronze é arquivo de auditoria |
| Particionamento por `run_ts` | Bronze | Permite reprocessar uma janela específica sem re-ingerir tudo |